In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
import mlflow
import mlflow.pytorch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"MLflow version: {mlflow.__version__}")

/Users/marwanamr/MLOPS course/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Using device: cpu
MLflow version: 3.1.4


In [ ]:
DATA_PATH = "../Assignment1/archive/fashion-mnist_train.csv"
TEST_PATH = "../Assignment1/archive/fashion-mnist_test.csv"

def load_fashion_mnist(train_path, test_path):
    df_train = pd.read_csv(train_path)
    df_test  = pd.read_csv(test_path)


    X_train = df_train.iloc[:, 1:].values.astype(np.float32) / 255.0
    y_train = df_train.iloc[:, 0].values.astype(np.int64)
    X_test  = df_test.iloc[:, 1:].values.astype(np.float32) / 255.0
    y_test  = df_test.iloc[:, 0].values.astype(np.int64)

    X_train = X_train.reshape(-1, 1, 28, 28)
    X_test  = X_test.reshape(-1, 1, 28, 28)

    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_ds  = TensorDataset(torch.tensor(X_test),  torch.tensor(y_test))
    return train_ds, test_ds

train_dataset, test_dataset = load_fashion_mnist(DATA_PATH, TEST_PATH)
print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")

Train samples: 60000 | Test samples: 10000


In [ ]:
class FashionCNN(nn.Module):
    """Compact CNN for Fashion-MNIST classification (10 classes)."""

    def __init__(self, dropout_rate=0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), 
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                             

            nn.Conv2d(32, 64, kernel_size=3, padding=1), 
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                           
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


dummy = torch.randn(4, 1, 28, 28)
out = FashionCNN()(dummy)
print(f"Output shape: {out.shape}") 

Output shape: torch.Size([4, 10])


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct    += (logits.argmax(1) == y).sum().item()
        total      += len(y)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = model(X)
        loss = criterion(logits, y)
        total_loss += loss.item() * len(y)
        correct    += (logits.argmax(1) == y).sum().item()
        total      += len(y)
    return total_loss / total, correct / total


def run_experiment(
    learning_rate: float,
    batch_size: int,
    epochs: int,
    dropout_rate: float = 0.25,
    optimizer_name: str = "adam",
    run_name: str = None,
):
    """Single training run fully instrumented with MLflow."""

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

    with mlflow.start_run(run_name=run_name):

        mlflow.log_params({
            "learning_rate":  learning_rate,
            "batch_size":     batch_size,
            "epochs":         epochs,
            "dropout_rate":   dropout_rate,
            "optimizer":      optimizer_name,
            "model":          "FashionCNN",
        })

        mlflow.set_tag("student_id", "202201743") 
        mlflow.set_tag("dataset",    "Fashion-MNIST")
        mlflow.set_tag("framework",  "PyTorch")

        model     = FashionCNN(dropout_rate=dropout_rate).to(DEVICE)
        criterion = nn.CrossEntropyLoss()

        if optimizer_name == "adam":
            optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        elif optimizer_name == "sgd":
            optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
        else:
            raise ValueError(f"Unknown optimizer: {optimizer_name}")

        best_val_acc = 0.0

        for epoch in range(1, epochs + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
            val_loss,   val_acc   = evaluate(model, test_loader, criterion)

            mlflow.log_metrics({
                "train_loss":     round(train_loss, 6),
                "train_accuracy": round(train_acc,  6),
                "val_loss":       round(val_loss,   6),
                "val_accuracy":   round(val_acc,    6),
            }, step=epoch)

            if val_acc > best_val_acc:
                best_val_acc = val_acc

            print(f"  Epoch {epoch:02d}/{epochs} "
                  f"| train_loss={train_loss:.4f}  train_acc={train_acc:.4f} "
                  f"| val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

        # Log best summary metric
        mlflow.log_metric("best_val_accuracy", best_val_acc)

        # ── Pillar 4: Model Artifact (MLflow Model Flavor) ────────────────
        mlflow.pytorch.log_model(
            model,
            artifact_path="model",
            # Registers conda/pip environment automatically
        )

        print(f"  >>> Best val accuracy: {best_val_acc:.4f}")
        print(f"  >>> Run logged to MLflow.\n")
        return best_val_acc

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("Assignment3_Marwan_Amr") 

configs = [
    (0.001, 64,  10, 0.25, "adam", "LR=0.001_BS=64_adam"),
    (0.01,  64,  10, 0.25, "adam", "LR=0.01_BS=64_adam"),
    (0.1,   64,  10, 0.25, "adam", "LR=0.1_BS=64_adam"),
    (0.001, 128, 10, 0.25, "adam", "LR=0.001_BS=128_adam"),
    (0.001, 64,  10, 0.25, "sgd",  "LR=0.001_BS=64_sgd"),
]

results = []
for lr, bs, ep, dr, opt, name in configs:
    print(f"\n{'='*60}")
    print(f"Starting run: {name}")
    print(f"{'='*60}")
    acc = run_experiment(
        learning_rate=lr,
        batch_size=bs,
        epochs=ep,
        dropout_rate=dr,
        optimizer_name=opt,
        run_name=name,
    )
    results.append({"run": name, "best_val_accuracy": acc})

2026/03/10 04:07:40 INFO mlflow.tracking.fluent: Experiment with name 'Assignment3_Marwan_Amr' does not exist. Creating a new experiment.



Starting run: LR=0.001_BS=64_adam
  Epoch 01/10 | train_loss=0.3987  train_acc=0.8546 | val_loss=0.2712  val_acc=0.8982
  Epoch 02/10 | train_loss=0.2764  train_acc=0.8991 | val_loss=0.2422  val_acc=0.9136
  Epoch 03/10 | train_loss=0.2377  train_acc=0.9119 | val_loss=0.2252  val_acc=0.9181
  Epoch 04/10 | train_loss=0.2101  train_acc=0.9231 | val_loss=0.2227  val_acc=0.9182
  Epoch 05/10 | train_loss=0.1867  train_acc=0.9301 | val_loss=0.2114  val_acc=0.9198
  Epoch 06/10 | train_loss=0.1676  train_acc=0.9370 | val_loss=0.2205  val_acc=0.9197
  Epoch 07/10 | train_loss=0.1491  train_acc=0.9452 | val_loss=0.2196  val_acc=0.9239
  Epoch 08/10 | train_loss=0.1310  train_acc=0.9512 | val_loss=0.2082  val_acc=0.9273
  Epoch 09/10 | train_loss=0.1170  train_acc=0.9558 | val_loss=0.2259  val_acc=0.9254


2026/03/10 04:11:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Epoch 10/10 | train_loss=0.1048  train_acc=0.9607 | val_loss=0.2084  val_acc=0.9308


2026/03/10 04:11:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

  >>> Best val accuracy: 0.9352
  >>> Run logged to MLflow.

🏃 View run LR=0.001_BS=128_adam at: http://127.0.0.1:5001/#/experiments/442082645821672869/runs/089d4a3a742f4283a01c5c33234885c4
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/442082645821672869

Starting run: LR=0.001_BS=64_sgd
  Epoch 01/10 | train_loss=0.5396  train_acc=0.8133 | val_loss=0.3429  val_acc=0.8808
  Epoch 02/10 | train_loss=0.3394  train_acc=0.8789 | val_loss=0.3048  val_acc=0.8886
  Epoch 03/10 | train_loss=0.2961  train_acc=0.8940 | val_loss=0.2685  val_acc=0.9040
  Epoch 04/10 | train_loss=0.2701  train_acc=0.9024 | val_loss=0.2497  val_acc=0.9110
  Epoch 05/10 | train_loss=0.2528  train_acc=0.9092 | val_loss=0.2430  val_acc=0.9124
  Epoch 06/10 | train_loss=0.2341  train_acc=0.9165 | val_loss=0.2523  val_acc=0.9090
  Epoch 07/10 | train_loss=0.2215  train_acc=0.9205 | val_loss=0.2292  val_acc=0.9186
  Epoch 08/10 | train_loss=0.2108  train_acc=0.9241 | val_loss=0.2322  val_acc=0.9160
  Epoch 09/

2026/03/10 04:28:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Epoch 10/10 | train_loss=0.1901  train_acc=0.9327 | val_loss=0.2186  val_acc=0.9213


2026/03/10 04:28:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


  >>> Best val accuracy: 0.9213
  >>> Run logged to MLflow.

🏃 View run LR=0.001_BS=64_sgd at: http://127.0.0.1:5001/#/experiments/442082645821672869/runs/ad01eade0cbe4dc2aeb0ea64ff020228
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/442082645821672869
